# 04_feature_engineering
Merge features with turbidity labels and provide scaling utilities.


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler


In [ ]:
# Load processed features and metadata
X = pd.read_parquet('../data/processed/X_features.parquet')
meta = pd.read_parquet('../data/processed/meta.parquet')
df_y = pd.read_csv('../data/datasets/turbidity_ctd_satellite.csv')
df_y = df_y.rename(columns={'station': 'ctd'})
df_y['date'] = pd.to_datetime(df_y['date']).dt.date
meta['date'] = pd.to_datetime(meta['date']).dt.date
meta['ctd'] = meta['ctd'].astype(str)
df_y['ctd'] = df_y['ctd'].astype(str)
# Join features and labels
X_meta = pd.concat([meta.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
merged = X_meta.merge(df_y, on=['ctd','date'], how='inner')
print('merged shape', merged.shape)


In [ ]:
def standardize_features(df, feature_cols=None):
    scaler = StandardScaler()
    if feature_cols is None:
        # choose numeric columns excluding metadata/label columns
        feature_cols = df.select_dtypes(include=[float, int]).columns.tolist()
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.fit_transform(df_scaled[feature_cols])
    return df_scaled, scaler

# Example: do not scale meta columns
feature_cols = merged.columns.difference(['ctd','date','turbidity']).tolist()
merged_scaled, scaler = standardize_features(merged, feature_cols)
# Save datasets
save_dir = Path('../data/processed')
merged.to_parquet(save_dir / 'dataset_final.parquet', index=False)
merged_scaled.to_parquet(save_dir / 'dataset_final_scaled.parquet', index=False)
print('Saved dataset_final.parquet and dataset_final_scaled.parquet')
